# NovaRetail 360 · Clase 2 · Análisis exploratorio con Python · VERSIÓN DOCENTE (resuelta)
**Inteligencia de Negocios · Ing. Nancy Gélvez Garcia PhD. · UDistrital**

**Pregunta de la clase:** ¿qué está reduciendo los ingresos netos de NovaRetail 360: los descuentos, las devoluciones o la composición de las ventas?

**Cómo usar este cuaderno**
- Cada bloque gris es una *celda de código*. Para ejecutarla, haga clic dentro y presione **Mayús + Enter** (o el botón ▶ a su izquierda).
- Ejecute las celdas **en orden, de arriba hacia abajo**. Si una celda usa algo de una anterior que no se ejecutó, aparece un error.
- Esta versión ya tiene todo el código completo. La de los estudiantes tiene espacios `____` en las mismas celdas.

## 1. Cargar los datos (minutos 0 a 20)
Use la base SQLite de su equipo de la clase 1 (la que tiene las tablas del laboratorio 1 y la estrella).

**Para subirla a Colab:** ejecute la celda siguiente, haga clic en **Elegir archivos** y seleccione su archivo `.db`. Espere a que diga *100% done*.

In [ ]:
# Subir la base de datos a Colab
try:
    from google.colab import files
    subida = files.upload()                # abre el selector de archivos
    RUTA_BASE = list(subida.keys())[0]     # nombre del archivo que subió
except ImportError:
    RUTA_BASE = 'novaretail_estrella.db'   # fuera de Colab: archivo en la misma carpeta
print('Base que se va a usar:', RUTA_BASE)

In [ ]:
# Bibliotecas: pandas para tablas, sqlite3 para leer la base, matplotlib para gráficas
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')   # números con separador de miles

conexion = sqlite3.connect(RUTA_BASE)

# Leer las tablas del laboratorio 1 como DataFrames (tablas de pandas)
ventas       = pd.read_sql('SELECT * FROM ventas', conexion)
devoluciones = pd.read_sql('SELECT * FROM devoluciones', conexion)
clientes     = pd.read_sql('SELECT * FROM clientes', conexion)
productos    = pd.read_sql('SELECT * FROM productos', conexion)

print('ventas:', ventas.shape, '| devoluciones:', devoluciones.shape,
      '| clientes:', clientes.shape, '| productos:', productos.shape)

In [ ]:
# Primeras filas de ventas: así se ve un DataFrame
ventas.head()

In [ ]:
# Tipos de cada columna: los números deben ser int64 o float64, no object (texto)
ventas.dtypes

## 2. Preparar y combinar (minutos 20 a 70)
### 2.1 Filtrar y seleccionar

In [ ]:
# Seleccionar columnas y filtrar filas: ventas de la Tienda con descuento
con_descuento = ventas[(ventas['canal'] == 'Tienda') & (ventas['descuento'] > 0)]
con_descuento[['id_venta', 'fecha', 'cantidad', 'precio_unitario', 'descuento']].head()

In [ ]:
print('Ventas de Tienda con descuento:', len(con_descuento))

### 2.2 Combinar ventas con devoluciones, clientes y productos
Primero se suman las devoluciones **por venta** (igual que en SQL), para no repetir importes al combinar.

In [ ]:
devueltas_por_venta = devoluciones.groupby('id_venta', as_index=False)['cantidad_devuelta'].sum()

datos = (ventas
         .merge(devueltas_por_venta, on='id_venta', how='left')   # left: conserva todas las ventas
         .merge(clientes[['id_cliente', 'region', 'segmento']], on='id_cliente', how='left')
         .merge(productos[['id_producto', 'categoria']], on='id_producto', how='left'))

datos['cantidad_devuelta'] = datos['cantidad_devuelta'].fillna(0)   # sin devolución = 0
print(datos.shape)
datos.head()

### 2.3 Calcular las medidas
Del precio de lista a las ventas netas hay dos pérdidas: los **descuentos** y las **devoluciones**.

In [ ]:
datos['importe_bruto']   = datos['cantidad'] * datos['precio_unitario']
datos['valor_descuento'] = datos['importe_bruto'] * datos['descuento']
datos['importe_neto']    = datos['importe_bruto'] - datos['valor_descuento']
datos['valor_devuelto']  = datos['cantidad_devuelta'] * datos['precio_unitario'] * (1 - datos['descuento'])
datos['ventas_netas']    = datos['importe_neto'] - datos['valor_devuelto']
datos['periodo']         = datos['fecha'].str[:7]                # 'aaaa-mm'

datos[['id_venta', 'importe_bruto', 'valor_descuento', 'valor_devuelto', 'ventas_netas']].head()

### 2.4 Validar contra SQL
El total de ventas netas en Python debe ser **igual** al control C2 de la clase 1. Si no coincide, hay un error en la preparación.

In [ ]:
total_python = datos['ventas_netas'].sum()
total_sql = pd.read_sql('SELECT SUM(ventas_netas) AS total FROM FactVentas', conexion)['total'][0]

print(f'Ventas netas en Python: ${total_python:,.0f}')
print(f'Ventas netas en SQL:    ${total_sql:,.0f}')
print('¿Coinciden?', round(total_python) == round(total_sql))

## 3. Análisis exploratorio (minutos 80 a 125)
### 3.1 Estadística descriptiva

In [ ]:
datos[['cantidad', 'precio_unitario', 'descuento', 'ventas_netas']].describe()

### 3.2 ¿Cuánto se pierde entre el importe bruto y las ventas netas?

In [ ]:
bruto       = datos['importe_bruto'].sum()
descuentos  = datos['valor_descuento'].sum()
devuelto    = datos['valor_devuelto'].sum()
netas       = datos['ventas_netas'].sum()
perdida     = bruto - netas

resumen = pd.DataFrame({
    'concepto': ['Importe bruto', 'Descuentos', 'Devoluciones', 'Ventas netas'],
    'valor':    [bruto, -descuentos, -devuelto, netas]})
print(resumen.to_string(index=False))
print(f'\nPérdida total: ${perdida:,.0f} ({perdida / bruto:.1%} del bruto)')
print(f'  Descuentos:   {descuentos / perdida:.1%} de la pérdida')
print(f'  Devoluciones: {devuelto / perdida:.1%} de la pérdida')

### 3.3 ¿Dónde se pierde más? Agrupar por categoría, canal y región
`groupby` agrupa las filas y `agg` calcula los totales de cada grupo. Los porcentajes se calculan **después**, dividiendo totales (nunca promediando porcentajes).

In [ ]:
def perdida_por(columna):
    t = datos.groupby(columna).agg(
        importe_bruto=('importe_bruto', 'sum'),
        descuentos=('valor_descuento', 'sum'),
        devoluciones=('valor_devuelto', 'sum'),
        ventas_netas=('ventas_netas', 'sum'),
        n_ventas=('id_venta', 'count'))
    t['pct_descuento']    = t['descuentos'] / t['importe_bruto'] * 100
    t['pct_devolucion']   = t['devoluciones'] / t['importe_bruto'] * 100
    t['pct_perdida']      = t['pct_descuento'] + t['pct_devolucion']
    t['participacion_vn'] = t['ventas_netas'] / t['ventas_netas'].sum() * 100
    return t.sort_values('pct_perdida', ascending=False)

perdida_por('categoria')

In [ ]:
perdida_por('canal')

In [ ]:
perdida_por('region')

### 3.4 ¿El descuento hace vender más unidades?

In [ ]:
por_descuento = datos.groupby('descuento').agg(
    ventas=('id_venta', 'count'),
    cantidad_promedio=('cantidad', 'mean'))
print(por_descuento)
print('\nCorrelación entre descuento y cantidad:', round(datos['descuento'].corr(datos['cantidad']), 3))

### 3.5 Valores atípicos (regla del rango intercuartílico)
Un valor es atípico si está por encima de Q3 + 1,5 × (Q3 − Q1). **Atípico no significa error:** hay que revisar qué ventas son antes de decidir.

In [ ]:
q1 = datos['ventas_netas'].quantile(0.25)
q3 = datos['ventas_netas'].quantile(0.75)
limite = q3 + 1.5 * (q3 - q1)
atipicas = datos[datos['ventas_netas'] > limite]

print(f'Q1 = ${q1:,.0f}   Q3 = ${q3:,.0f}   Límite = ${limite:,.0f}')
print('Ventas atípicas:', len(atipicas))
atipicas[['id_venta', 'id_producto', 'categoria', 'cantidad', 'precio_unitario', 'descuento', 'ventas_netas']].sort_values('ventas_netas', ascending=False).head(10)

## 4. Visualizaciones (minutos 125 a 160)
Cada gráfica responde una pregunta. Escriba debajo de cada una, en una frase, qué muestra.

In [ ]:
# 4.1 Del bruto a las ventas netas (millones de pesos)
pasos   = ['Importe bruto', 'Descuentos', 'Devoluciones', 'Ventas netas']
alturas = [bruto, descuentos, devuelto, netas]
bases   = [0, bruto - descuentos, bruto - descuentos - devuelto, 0]
colores = ['#7A8794', '#C0504D', '#C0504D', '#2B5A97']

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(pasos, [a / 1e6 for a in alturas], bottom=[b / 1e6 for b in bases], color=colores)
for i, a in enumerate(alturas):
    ax.text(i, (bases[i] + a) / 1e6 + 5, f'{a / 1e6:,.1f}', ha='center')
ax.set_ylabel('Millones de pesos')
ax.set_title('De importe bruto a ventas netas: descuentos y devoluciones')
plt.tight_layout()
plt.show()

In [ ]:
# 4.2 Pérdida por categoría: descuento y devolución como % del bruto
t = perdida_por('categoria')[['pct_descuento', 'pct_devolucion']]
ax = t.plot(kind='barh', stacked=True, figsize=(8, 4), color=['#E8A33D', '#C0504D'])
ax.set_xlabel('% del importe bruto')
ax.set_title('Pérdida por categoría')
ax.legend(['Descuento', 'Devolución'])
plt.tight_layout()
plt.show()

In [ ]:
# 4.3 ¿Más descuento, más unidades? Cantidad promedio por nivel de descuento
ax = por_descuento['cantidad_promedio'].plot(kind='bar', figsize=(7, 4), color='#2B5A97')
ax.set_xticklabels([f'{d:.0%}' for d in por_descuento.index], rotation=0)
ax.set_xlabel('Descuento aplicado')
ax.set_ylabel('Unidades promedio por venta')
ax.set_ylim(0, 4.5)
ax.set_title('La cantidad vendida casi no cambia con el descuento')
plt.tight_layout()
plt.show()

In [ ]:
# 4.4 Distribución de las ventas netas por categoría (los puntos son atípicos)
datos.boxplot(column='ventas_netas', by='categoria', figsize=(8, 4.5))
plt.suptitle('')
plt.title('Ventas netas por venta, según categoría')
plt.ylabel('Pesos')
plt.tight_layout()
plt.show()

In [ ]:
# 4.5 Devoluciones por mes de la venta original
mensual = datos.groupby('periodo')[['valor_descuento', 'valor_devuelto']].sum() / 1e6
ax = mensual.plot(marker='o', figsize=(8, 4), color=['#E8A33D', '#C0504D'])
ax.set_ylabel('Millones de pesos')
ax.set_title('Descuentos y devoluciones por mes')
ax.legend(['Descuentos', 'Devoluciones'])
plt.tight_layout()
plt.show()

## 5. Hallazgos y limitaciones (minutos 160 a 180) · Respuesta de referencia
Cifras de la base de referencia (1.192 ventas). Las de cada equipo pueden variar según su limpieza del laboratorio 1.

**Hallazgo 1. Los descuentos pesan algo más que las devoluciones.** Entre el importe bruto ($736,3 millones) y las ventas netas ($668,8 millones) se pierden $67,5 millones (9,2 %). El 55,6 % son descuentos ($37,5 millones) y el 44,4 %, devoluciones ($30,0 millones). *Fuente: 3.2 y gráfica 4.1.*

**Hallazgo 2. Tecnología pierde más, y por devoluciones, no por descuentos.** Pierde el 10,4 % de su importe bruto: 6,0 % por devoluciones (la más alta) y 4,4 % por descuentos (la más baja). Hogar tiene el descuento más alto (6,0 %) pero pocas devoluciones (2,3 %). En canales, Tienda pierde más (10,3 %), sobre todo por devoluciones (5,1 %). *Fuente: 3.3 y gráfica 4.2.*

**Hallazgo 3. Los descuentos no están comprando volumen.** La cantidad promedio por venta es casi igual sin descuento (3,48 unidades) y con 15 % (3,43). La correlación es 0,003. *Fuente: 3.4 y gráfica 4.3.*

**Sobre la composición de las ventas:** las cuatro categorías aportan entre 23 % y 26 % de las ventas netas. La mezcla está equilibrada y no explica la pérdida: un cambio en la mezcla cambiaría el total, pero eso no sería en sí una pérdida.

**Limitaciones:**
- Datos sintéticos de un solo semestre.
- Correlación no es causalidad: que el descuento no se asocie con más unidades no prueba que no sirva (podría, por ejemplo, atraer clientes nuevos, algo que estos datos no miden).
- Las devoluciones se asignan al mes de la venta: las ventas de mayo y junio todavía podrían recibir devoluciones, por eso esos meses se ven mejor (gráfica 4.5).
- Las 20 ventas atípicas son ventas legítimas de productos caros en 6 unidades, no errores: no se eliminan.